# Advanced: Multi-Strategy DateTime Generalization with PAMOLA.CORE

**Goal**: Master all 4 datetime anonymization strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Rounding-based temporal precision reduction
- **Strategy 2**: Time binning and categorical grouping
- **Strategy 3**: Component-based date extraction
- **Strategy 4**: Relative time expressions
- Calculate advanced privacy metrics (k-anonymity, l-diversity)
- Analyze temporal information loss
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of privacy concepts
- Familiarity with operation.execute() pattern
---

## Step 1: Setup and Imports

**How to use:**
- Import all required libraries
- Set up paths and configurations
- Verify PAMOLA installation

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── anonymization/generalization/
        └── 02_datetime_generalization_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime, timedelta
import time
from IPython.display import Image, display

print("🔍 Detecting PAMOLA installation...\n")

current_dir = Path.cwd()
print(f"📍 Notebook location: {current_dir}")

project_root = current_dir
for level in range(6):
    if (project_root / 'pamola_core').exists():
        print(f"✓ Found pamola_core at level {level}: {project_root}")
        break
    project_root = project_root.parent
else:
    raise FileNotFoundError("❌ Could not find pamola_core directory!")

sys.path.insert(0, str(project_root))

from pamola_core.anonymization.generalization.datetime_op import DateTimeGeneralizationOperation
from pamola_core.utils.ops.op_data_source import DataSource
from pamola_core.utils.progress import HierarchicalProgressTracker
from pamola_core.utils.tasks.task_reporting import TaskReporter

print("✅ All imports successful!")
print(f"📅 Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

## Step 2: Load Complex Dataset

In [ ]:
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'

if data_path.exists():
    df = pd.read_csv(data_path)
    for col in ['snapshot_date']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])
    print(f"✓ Loaded {len(df)} records")
else:
    print("📊 Generating 1000 records...")
    np.random.seed(42)
    n = 1000
    base = datetime(2024, 1, 1)
    
    login_times = [base + timedelta(days=np.random.randint(0, 180),
                                    hours=np.random.randint(0, 24),
                                    minutes=np.random.randint(0, 60)) for _ in range(n)]
    
    df = pd.DataFrame({
        'record_id': range(1, n + 1),
        'login_time': login_times,
        'transaction_time': [t + timedelta(minutes=np.random.randint(5, 60)) for t in login_times],
        'last_activity': [t + timedelta(hours=np.random.randint(1, 24)) for t in login_times],
        'created_at': [t - timedelta(days=np.random.randint(30, 365)) for t in login_times],
        'event_type': np.random.choice(['Purchase', 'View', 'Download', 'Upload'], n),
        'location': np.random.choice(['US-East', 'US-West', 'EU', 'Asia'], n),
        'user_type': np.random.choice(['Free', 'Premium', 'Enterprise'], n, p=[0.6, 0.3, 0.1]),
        'value': np.random.randint(10, 1000, n)
    })
    
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Saved to {data_path}")
    except PermissionError:
        print("⚠️  Cannot save (file may be open)")

print(f"\n📊 Dataset: {len(df):,} records, {len(df.columns)} columns")
print(df.head())

## Step 3: Setup Shared Environment

In [ ]:
# Field configuration
FIELD_CONFIG = {
    "strategy1_field": "snapshot_date",
    "strategy2_field": "snapshot_date",
    "strategy3_field": "snapshot_date",
    "strategy4_field": "snapshot_date"
}

# Validate
print("📋 Validating fields...\n")
for strategy, field in FIELD_CONFIG.items():
    if field not in df.columns:
        raise ValueError(f"Field '{field}' not found!")
    print(f"  ✓ {strategy:20s}: '{field}' ({df[field].nunique()} unique)")

# Setup
def create_config_kwargs():
    return {"dataset_name": "main_dataset"}

task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)

task_reporter = TaskReporter(
    task_id="datetime_adv_001",
    task_type="multi_strategy",
    description="Multi-strategy datetime generalization",
    report_path=task_dir
)

kwargs = create_config_kwargs()
data_source = DataSource(dataframes={"main_dataset": df})

print(f"\n✅ Environment ready!")

## STRATEGY 1: Rounding-Based Generalization

**How to use:**
- Reduces temporal precision by rounding to larger time units
- Removes exact timing information
- Preserves temporal patterns and ordering

**Key parameters:**
- `strategy='rounding'`
- `rounding_unit='day'`: Round to day level
- `timezone_handling='preserve'`: Keep original timezone
- `min_privacy_threshold=0.3`: Require 30% reduction

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: ROUNDING-BASED GENERALIZATION")
print("=" * 80 + "\n")

tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Rounding",
    unit="steps"
)

operation_s1 = DateTimeGeneralizationOperation(
    field_name=FIELD_CONFIG["strategy1_field"],
    mode='ENRICH',
    strategy='rounding',
    rounding_unit='day',
    output_field_name=f"{FIELD_CONFIG['strategy1_field']}_rounded",
    timezone_handling='preserve',
    min_privacy_threshold=0.3,
    null_strategy='PRESERVE',
    use_vectorization=True,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
start_time = time.time()

result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_rounding',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Load results
output_files_s1 = sorted(
    list((task_dir / 'strategy1_rounding' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s1:
    result_df_s1 = pd.read_csv(output_files_s1[0])
    for col in [FIELD_CONFIG["strategy1_field"], f"{FIELD_CONFIG['strategy1_field']}_rounded"]:
        if col in result_df_s1.columns:
            result_df_s1[col] = pd.to_datetime(result_df_s1[col])
    
    field_s1 = FIELD_CONFIG["strategy1_field"]
    output_field_s1 = f"{field_s1}_rounded"
    print(f"\n📊 Reduction: {result_df_s1[field_s1].nunique()} → {result_df_s1[output_field_s1].nunique()} unique timestamps")

## STRATEGY 2: Time Binning Generalization

**How to use:**
- Groups timestamps into categorical time ranges
- Creates meaningful time periods (morning, afternoon, etc.)
- Converts continuous time to discrete categories

**Key parameters:**
- `strategy='binning'`
- `bin_type='business_period'`: Group by business hours
- `interval_size=6`: Hour range size
- `min_privacy_threshold=0.5`: Higher threshold for categorical

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: TIME BINNING GENERALIZATION")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Binning",
    unit="steps"
)

operation_s2 = DateTimeGeneralizationOperation(
    field_name=FIELD_CONFIG["strategy2_field"],
    mode='ENRICH',
    strategy='binning',
    bin_type='business_period',
    interval_size=6,
    output_field_name=f"{FIELD_CONFIG['strategy2_field']}_binned",
    timezone_handling='preserve',
    min_privacy_threshold=0.5,
    null_strategy='PRESERVE',
    use_vectorization=True,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_binning',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Load results
output_files_s2 = sorted(
    list((task_dir / 'strategy2_binning' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s2:
    result_df_s2 = pd.read_csv(output_files_s2[0])
    # Convert datetime columns
    for col in result_df_s2.columns:
        if 'time' in col.lower() or 'created' in col.lower() or 'activity' in col.lower():
            try:
                result_df_s2[col] = pd.to_datetime(result_df_s2[col])
            except:
                pass
    
    field_s2 = FIELD_CONFIG["strategy2_field"]
    output_field_s2 = f"{field_s2}_binned"
    if output_field_s2 in result_df_s2.columns:
        print(f"\n📊 Time periods: {result_df_s2[output_field_s2].unique()}")

## STRATEGY 3: Component-Based Generalization

**How to use:**
- Extracts specific datetime components only
- Removes unwanted temporal details
- Useful for keeping year-month but removing day/time

**Key parameters:**
- `strategy='component'`
- `keep_components=['year', 'month']`: Extract only year and month
- `strftime_output_format='%Y-%m'`: Format as YYYY-MM
- `min_privacy_threshold=0.4`: Require 40% reduction

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: COMPONENT-BASED GENERALIZATION")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Component",
    unit="steps"
)

operation_s3 = DateTimeGeneralizationOperation(
    field_name=FIELD_CONFIG["strategy3_field"],
    mode='ENRICH',
    strategy='component',
    keep_components=['year', 'month'],
    strftime_output_format='%Y-%m',
    output_field_name=f"{FIELD_CONFIG['strategy3_field']}_component",
    timezone_handling='preserve',
    min_privacy_threshold=0.4,
    null_strategy='PRESERVE',
    use_vectorization=True,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
print(f"  Components to keep: {operation_s3.keep_components}")
print(f"  Output format: {operation_s3.strftime_output_format}")

start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_component',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Load results
output_files_s3 = sorted(
    list((task_dir / 'strategy3_component' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s3:
    result_df_s3 = pd.read_csv(output_files_s3[0])
    
    # Convert datetime columns (except component output which is string)
    for col in result_df_s3.columns:
        if col == f"{FIELD_CONFIG['strategy3_field']}_component":
            continue  # Keep as string
        if 'time' in col.lower() or 'created' in col.lower() or 'activity' in col.lower():
            try:
                result_df_s3[col] = pd.to_datetime(result_df_s3[col])
            except:
                pass
    
    field_s3 = FIELD_CONFIG["strategy3_field"]
    output_field_s3 = f"{field_s3}_component"
    
    if output_field_s3 in result_df_s3.columns:
        print(f"\n📊 Original unique: {result_df_s3[field_s3].nunique()}")
        print(f"📊 Component unique: {result_df_s3[output_field_s3].nunique()}")
        print(f"\n📅 Sample components: {result_df_s3[output_field_s3].unique()[:5].tolist()}")

## STRATEGY 4: Relative Time Expressions

**How to use:**
- Converts absolute timestamps to relative descriptions
- Groups times relative to reference date
- Useful for expressing temporal distance without exact dates

**Key parameters:**
- `strategy='relative'`
- `reference_date=None`: Use current date as reference
- Categories: "Days ago", "Weeks ago", "Months ago", "More than a year ago"
- `min_privacy_threshold=0.6`: Highest threshold for categorical grouping

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 4: RELATIVE TIME EXPRESSIONS")
print("=" * 80 + "\n")

tracker_s4 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 4: Relative",
    unit="steps"
)

operation_s4 = DateTimeGeneralizationOperation(
    field_name=FIELD_CONFIG["strategy4_field"],
    mode='ENRICH',
    strategy='relative',
    reference_date="2024-07-01T00:00:00",
    output_field_name=f"{FIELD_CONFIG['strategy4_field']}_relative",
    timezone_handling='preserve',
    min_privacy_threshold=0.6,
    null_strategy='PRESERVE',
    use_vectorization=True,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 4 configured")
print(f"  Reference date: {operation_s4.reference_date.strftime('%Y-%m-%d')}")

start_time = time.time()

result_s4 = operation_s4.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy4_relative',
    reporter=task_reporter,
    progress_tracker=tracker_s4,
    **kwargs
)

elapsed_s4 = time.time() - start_time
print(f"\n✅ Strategy 4 completed in {elapsed_s4:.2f}s")

# Load results
output_files_s4 = sorted(
    list((task_dir / 'strategy4_relative' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s4:
    result_df_s4 = pd.read_csv(output_files_s4[0])
    
    # Convert datetime columns (except relative output which is string)
    for col in result_df_s4.columns:
        if col == f"{FIELD_CONFIG['strategy4_field']}_relative":
            continue  # Keep as string
        if 'time' in col.lower() or 'created' in col.lower() or 'activity' in col.lower():
            try:
                result_df_s4[col] = pd.to_datetime(result_df_s4[col])
            except:
                pass
    
    field_s4 = FIELD_CONFIG["strategy4_field"]
    output_field_s4 = f"{field_s4}_relative"
    
    if output_field_s4 in result_df_s4.columns:
        print(f"\n📊 Relative categories: {result_df_s4[output_field_s4].unique().tolist()}")
        print(f"\n📈 Distribution:")
        print(result_df_s4[output_field_s4].value_counts())

## Strategy Comparison

Compare performance and effectiveness across all 4 strategies

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

# Check which strategies completed
strategies = []
elapsed_times = []

if 'elapsed_s1' in globals():
    strategies.append('Strategy 1: Rounding')
    elapsed_times.append(elapsed_s1)
if 'elapsed_s2' in globals():
    strategies.append('Strategy 2: Binning')
    elapsed_times.append(elapsed_s2)
if 'elapsed_s3' in globals():
    strategies.append('Strategy 3: Component')
    elapsed_times.append(elapsed_s3)
if 'elapsed_s4' in globals():
    strategies.append('Strategy 4: Relative')
    elapsed_times.append(elapsed_s4)

if strategies:
    print("⏱️  Execution Time:")
    for strategy, elapsed in zip(strategies, elapsed_times):
        print(f"  {strategy:30s} {elapsed:6.2f}s")
    print(f"  {'Total:':<30s} {sum(elapsed_times):6.2f}s")
    
    # Find fastest and slowest
    if len(elapsed_times) > 1:
        fastest_idx = elapsed_times.index(min(elapsed_times))
        slowest_idx = elapsed_times.index(max(elapsed_times))
        print(f"\n  ⚡ Fastest: {strategies[fastest_idx]}")
        print(f"  🐌 Slowest: {strategies[slowest_idx]}")
        print(f"  ⚖️  Difference: {max(elapsed_times) - min(elapsed_times):.2f}s")

# Compare privacy improvements
print("\n📊 Privacy Reduction:")
print("-" * 60)

privacy_data = []

if 'result_df_s1' in globals():
    field = FIELD_CONFIG["strategy1_field"]
    output = f"{field}_rounded"
    if output in result_df_s1.columns:
        orig = result_df_s1[field].nunique()
        gen = result_df_s1[output].nunique()
        reduction = (1 - gen/orig) * 100
        privacy_data.append(('Strategy 1', orig, gen, reduction))

if 'result_df_s2' in globals():
    field = FIELD_CONFIG["strategy2_field"]
    output = f"{field}_binned"
    if output in result_df_s2.columns:
        orig = result_df_s2[field].nunique()
        gen = result_df_s2[output].nunique()
        reduction = (1 - gen/orig) * 100
        privacy_data.append(('Strategy 2', orig, gen, reduction))

if 'result_df_s3' in globals():
    field = FIELD_CONFIG["strategy3_field"]
    output = f"{field}_component"
    if output in result_df_s3.columns:
        orig = result_df_s3[field].nunique()
        gen = result_df_s3[output].nunique()
        reduction = (1 - gen/orig) * 100
        privacy_data.append(('Strategy 3', orig, gen, reduction))

if 'result_df_s4' in globals():
    field = FIELD_CONFIG["strategy4_field"]
    output = f"{field}_relative"
    if output in result_df_s4.columns:
        orig = result_df_s4[field].nunique()
        gen = result_df_s4[output].nunique()
        reduction = (1 - gen/orig) * 100
        privacy_data.append(('Strategy 4', orig, gen, reduction))

if privacy_data:
    print(f"{'Strategy':<15} {'Original':<10} {'Generalized':<12} {'Reduction'}")
    print("-" * 60)
    for strategy, orig, gen, reduction in privacy_data:
        print(f"{strategy:<15} {orig:<10} {gen:<12} {reduction:>6.1f}%")
    
    # Best privacy protection
    best_idx = max(range(len(privacy_data)), key=lambda i: privacy_data[i][3])
    print(f"\n🔒 Best privacy: {privacy_data[best_idx][0]} ({privacy_data[best_idx][3]:.1f}% reduction)")

print("\n" + "=" * 80)

## Sample Data Comparison

Show before/after examples from each strategy

In [ ]:
print("\n" + "=" * 80)
print("📋 SAMPLE DATA COMPARISON")
print("=" * 80)

# Show first 5 records from each strategy
sample_size = 5

if 'result_df_s1' in globals():
    print("\n1️⃣ STRATEGY 1: ROUNDING")
    field = FIELD_CONFIG["strategy1_field"]
    output = f"{field}_rounded"
    cols = [ field, output]
    existing_cols = [c for c in cols if c in result_df_s1.columns]
    print(result_df_s1[existing_cols].head(sample_size))

if 'result_df_s2' in globals():
    print("\n2️⃣ STRATEGY 2: BINNING")
    field = FIELD_CONFIG["strategy2_field"]
    output = f"{field}_binned"
    cols = [field, output]
    existing_cols = [c for c in cols if c in result_df_s2.columns]
    print(result_df_s2[existing_cols].head(sample_size))

if 'result_df_s3' in globals():
    print("\n3️⃣ STRATEGY 3: COMPONENT")
    field = FIELD_CONFIG["strategy3_field"]
    output = f"{field}_component"
    cols = [field, output]
    existing_cols = [c for c in cols if c in result_df_s3.columns]
    print(result_df_s3[existing_cols].head(sample_size))

if 'result_df_s4' in globals():
    print("\n4️⃣ STRATEGY 4: RELATIVE")
    field = FIELD_CONFIG["strategy4_field"]
    output = f"{field}_relative"
    cols = [field, output]
    existing_cols = [c for c in cols if c in result_df_s4.columns]
    print(result_df_s4[existing_cols].head(sample_size))

print("\n" + "=" * 80)

## DETAILED ARTIFACT REVIEW

Review artifacts from all 4 strategies with newest files

In [ ]:
strategy_dirs = [
    ('strategy1_rounding', 'Strategy 1: Rounding-Based'),
    ('strategy2_binning', 'Strategy 2: Time Binning'),
    ('strategy3_component', 'Strategy 3: Component-Based'),
    ('strategy4_relative', 'Strategy 4: Relative Time')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        print(f"\n⚠️  {title}: Directory not found")
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. METRICS JSON (NEWEST)
    print("\n1️⃣ METRICS (JSON):")
    print("-" * 80)
    
    metrics_dir = strategy_dir / 'metrics'
    if not metrics_dir.exists():
        print(f"⚠️  Metrics directory not found: {metrics_dir}")
    else:
        metrics_files = list(metrics_dir.glob('*.json'))
        
        if not metrics_files:
            print("⚠️  No metrics files found")
        else:
            # 1) Exclude data_types inside IF
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                # Use only filtered files
                target_files = filtered
                print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
            else:
                # Fallback: nothing left after filtering → use all
                target_files = metrics_files
                print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

            # 2) Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]

            print(f"📄 Loading LATEST: {latest_metrics_file.name}")
            mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
            print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
            print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
            print("=" * 80)
            
            try:
                with open(latest_metrics_file, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                
                # Extract metadata and metrics
                metadata = data.get('metadata', {})
                metrics = data.get('metrics', {})
                
                # Display metadata
                print("📋 METADATA:")
                print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
                print(f"   Name: {metadata.get('name', 'N/A')}")
                if 'operation' in metadata:
                    op = metadata['operation']
                    print(f"   Operation: {op.get('class', 'N/A')}.{op.get('function', 'N/A')}")
                
                # Display effectiveness metrics
                if 'effectiveness' in metrics:
                    print("\n📈 EFFECTIVENESS:")
                    eff = metrics['effectiveness']
                    print(f"   Total Records: {eff.get('total_records', 'N/A'):,}")
                    print(f"   Original Unique: {eff.get('original_unique', 'N/A')}")
                    print(f"   Anonymized Unique: {eff.get('anonymized_unique', 'N/A')}")
                    print(f"   Effectiveness Ratio: {eff.get('effectiveness_ratio', 0):.4f}")
                    print(f"   Null Increase: {eff.get('null_increase', 0):.4f}")
                
                # Display performance metrics  
                print("\n⚡ PERFORMANCE:")
                print(f"   Duration: {metrics.get('duration_seconds', 0):.4f}s")
                print(f"   Records Processed: {metrics.get('records_processed', 0):,}")
                print(f"   Records/Second: {metrics.get('records_per_second', 0):.2f}")
                if 'chunk_count' in metrics:
                    print(f"   Chunk Count: {metrics.get('chunk_count', 0)}")
                if 'chunk_size_used' in metrics:
                    print(f"   Chunk Size: {metrics.get('chunk_size_used', 0):,}")
                
                # Display datetime-specific metrics
                print("\n🕐 DATETIME-SPECIFIC METRICS:")
                datetime_keys = [
                    ('datetime_strategy', 'Strategy'),
                    ('rounding_unit', 'Rounding Unit'),
                    ('bin_type', 'Bin Type'),
                    ('unique_patterns_before', 'Patterns Before'),
                    ('unique_patterns_after', 'Patterns After'),
                    ('privacy_reduction_ratio', 'Privacy Reduction'),
                    ('original_date_span_days', 'Date Span (days)')
                ]
                
                for key, label in datetime_keys:
                    if key in metrics:
                        value = metrics[key]
                        if isinstance(value, float) and 0 < value < 1:
                            print(f"   {label}: {value:.2%}")
                        else:
                            print(f"   {label}: {value}")
                
                # Display other key metrics
                print("\n📌 OTHER METRICS:")
                other_keys = [
                    ('processing_rate', 'Processing Rate'),
                    ('adaptive_chunk_size', 'Adaptive Chunking'),
                    ('filtered_records', 'Filtered Records')
                ]
                
                for key, label in other_keys:
                    if key in metrics:
                        value = metrics[key]
                        if isinstance(value, float) and value < 1:
                            print(f"   {label}: {value:.2%}")
                        elif isinstance(value, bool):
                            print(f"   {label}: {'Yes' if value else 'No'}")
                        else:
                            print(f"   {label}: {value}")
            
            except json.JSONDecodeError as e:
                print(f"❌ JSON decode error: {e}")
            except Exception as e:
                print(f"❌ Error reading metrics: {e}")
    
    # 2. OUTPUT DATA SAMPLE (NEWEST FILE - First 10 rows)
    print("\n\n2️⃣ OUTPUT DATA (First 10 rows):")
    print("-" * 80)
    
    output_dir = strategy_dir / 'output'
    if not output_dir.exists():
        print(f"⚠️  Output directory not found: {output_dir}")
    else:
        output_files = sorted(list(output_dir.glob('*.csv')),
                            key=lambda x: x.stat().st_mtime, reverse=True)
        
        if not output_files:
            print("⚠️  No output files found")
        else:
            latest_output_file = output_files[0]
            
            # Show file info
            mtime = datetime.fromtimestamp(latest_output_file.stat().st_mtime)
            print(f"✓ Found {len(output_files)} output file(s)")
            print(f"📄 Loading LATEST: {latest_output_file.name}")
            print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
            print(f"   Size: {latest_output_file.stat().st_size / 1024:.1f} KB\n")
            
            try:
                output_df = pd.read_csv(latest_output_file)
                
                # Convert datetime columns
                for col in output_df.columns:
                    if 'time' in col.lower() or 'created' in col.lower() or 'activity' in col.lower():
                        # Skip if it's the binned/relative output (string format)
                        if 'binned' in col or 'relative' in col or 'component' in col:
                            continue
                        try:
                            output_df[col] = pd.to_datetime(output_df[col])
                        except:
                            pass
                
                print(f"📊 Shape: {output_df.shape[0]} rows × {output_df.shape[1]} columns")
                print(f"\n{output_df.head(10).to_string()}")
                
                # Show unique values for generalized field
                generalized_cols = [c for c in output_df.columns 
                                   if any(x in c for x in ['rounded', 'binned', 'component', 'relative'])]
                
                if generalized_cols:
                    gen_col = generalized_cols[0]
                    print(f"\n\n📊 {gen_col} - Unique Values:")
                    print("-" * 80)
                    unique_vals = output_df[gen_col].unique()
                    print(f"Total unique: {len(unique_vals)}")
                    print(f"\nSample values (first 10):")
                    for val in unique_vals[:10]:
                        count = (output_df[gen_col] == val).sum()
                        pct = (count / len(output_df)) * 100
                        print(f"  {str(val):<40} {count:>5} records ({pct:>5.1f}%)")
                    if len(unique_vals) > 10:
                        print(f"  ... and {len(unique_vals) - 10} more unique values")
            
            except Exception as e:
                print(f"❌ Error reading output: {e}")
    
    # 3. VISUALIZATIONS (NEWEST BATCH)
    print("\n\n3️⃣ VISUALIZATIONS:")
    print("-" * 80)
    
    viz_dir = strategy_dir / 'visualizations'
    if not viz_dir.exists():
        print(f"⚠️  Visualizations directory not found: {viz_dir}")
    else:
        viz_files = sorted(list(viz_dir.glob('*.png')),
                          key=lambda x: x.stat().st_mtime, reverse=True)
        
        if not viz_files:
            print("⚠️  No visualizations found")
        else:
            latest_time = viz_files[0].stat().st_mtime
            time_threshold = 10  # seconds
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            # Sort batch by name for consistent ordering
            latest_viz_batch.sort(key=lambda x: x.name)
            
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            # Display each visualization from latest batch
            for i, viz_file in enumerate(latest_viz_batch[:3], 1):  # Show first 3
                # Extract readable name from filename
                name_parts = viz_file.stem.split('_')
                if len(name_parts) > 3:
                    viz_name = ' '.join(name_parts[2:-1]).strip().title()
                else:
                    viz_name = viz_file.stem.replace('_', ' ').title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
            
            # Show info about remaining visualizations
            if len(latest_viz_batch) > 3:
                print(f"\n💡 Note: {len(latest_viz_batch) - 3} more visualization(s) in this batch not shown")
            if len(viz_files) > len(latest_viz_batch):
                print(f"💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

## Calculate Privacy Metrics

Calculate k-anonymity for original and generalized data

In [ ]:
print("\n" + "=" * 80)
print("🔒 PRIVACY METRICS CALCULATION")
print("=" * 80 + "\n")

def calculate_k_anonymity(df, quasi_identifiers):
    """Calculate k-anonymity metrics"""
    # Filter to only existing columns
    existing_qis = [q for q in quasi_identifiers if q in df.columns]
    if not existing_qis:
        return None
    
    group_sizes = df.groupby(existing_qis).size()
    return {
        'min_k': int(group_sizes.min()),
        'avg_k': float(group_sizes.mean()),
        'max_k': int(group_sizes.max()),
        'vulnerable_groups': int((group_sizes < 5).sum()),
        'total_groups': len(group_sizes)
    }

# Original data k-anonymity
try:
    # Use timestamp fields as quasi-identifiers
    original_qis = [FIELD_CONFIG["strategy1_field"], FIELD_CONFIG["strategy2_field"]]
    k_orig = calculate_k_anonymity(df, original_qis)
    
    if k_orig:
        print("📊 ORIGINAL DATA:")
        print(f"   Min k-anonymity: {k_orig['min_k']}")
        print(f"   Avg k-anonymity: {k_orig['avg_k']:.2f}")
        print(f"   Max k-anonymity: {k_orig['max_k']}")
        print(f"   Vulnerable groups (k<5): {k_orig['vulnerable_groups']}")
        print(f"   Total groups: {k_orig['total_groups']}")
    
    # After generalization (using Strategy 1 & 2 results)
    if 'result_df_s2' in globals():
        field1_gen = f"{FIELD_CONFIG['strategy1_field']}_rounded"
        field2_gen = f"{FIELD_CONFIG['strategy2_field']}_binned"
        gen_qis = [field1_gen, field2_gen]
        
        k_gen = calculate_k_anonymity(result_df_s2, gen_qis)
        
        if k_gen:
            print("\n✨ AFTER GENERALIZATION (Strategy 1 + 2):")
            print(f"   Min k-anonymity: {k_gen['min_k']} ({k_gen['min_k']/k_orig['min_k']:.1f}x improvement)")
            print(f"   Avg k-anonymity: {k_gen['avg_k']:.2f} ({k_gen['avg_k']/k_orig['avg_k']:.1f}x improvement)")
            print(f"   Max k-anonymity: {k_gen['max_k']}")
            print(f"   Vulnerable groups (k<5): {k_gen['vulnerable_groups']}")
            print(f"   Total groups: {k_gen['total_groups']}")
            
            print(f"\n🎯 Dataset is {k_gen['min_k']}-anonymous")
            
            if k_gen['vulnerable_groups'] == 0:
                print("✅ No vulnerable groups remaining!")
            else:
                improvement = (1 - k_gen['vulnerable_groups']/k_orig['vulnerable_groups']) * 100
                print(f"📉 Vulnerable groups reduced by {improvement:.1f}%")
        
except NameError:
    print("⚠️  FIELD_CONFIG not defined!")
    print("   Please run Part 1 first")
except Exception as e:
    print(f"⚠️  Error calculating privacy metrics: {e}")

print("\n" + "=" * 80)

## Information Loss Analysis

Compare temporal information loss across strategies

In [ ]:
print("\n" + "=" * 80)
print("📉 INFORMATION LOSS ANALYSIS")
print("=" * 80 + "\n")

info_loss_data = []

# Strategy 1: Rounding
if 'result_df_s1' in globals():
    field = FIELD_CONFIG["strategy1_field"]
    output = f"{field}_rounded"
    if output in result_df_s1.columns:
        orig_entropy = -sum(result_df_s1[field].value_counts(normalize=True) * 
                          np.log2(result_df_s1[field].value_counts(normalize=True)))
        gen_entropy = -sum(result_df_s1[output].value_counts(normalize=True) * 
                         np.log2(result_df_s1[output].value_counts(normalize=True)))
        entropy_loss = (orig_entropy - gen_entropy) / orig_entropy * 100
        info_loss_data.append(('Strategy 1: Rounding', orig_entropy, gen_entropy, entropy_loss))

# Strategy 2: Binning
if 'result_df_s2' in globals():
    field = FIELD_CONFIG["strategy2_field"]
    output = f"{field}_binned"
    if output in result_df_s2.columns:
        orig_entropy = -sum(result_df_s2[field].value_counts(normalize=True) * 
                          np.log2(result_df_s2[field].value_counts(normalize=True)))
        gen_entropy = -sum(result_df_s2[output].value_counts(normalize=True) * 
                         np.log2(result_df_s2[output].value_counts(normalize=True)))
        entropy_loss = (orig_entropy - gen_entropy) / orig_entropy * 100
        info_loss_data.append(('Strategy 2: Binning', orig_entropy, gen_entropy, entropy_loss))

# Strategy 3: Component
if 'result_df_s3' in globals():
    field = FIELD_CONFIG["strategy3_field"]
    output = f"{field}_component"
    if output in result_df_s3.columns:
        orig_entropy = -sum(result_df_s3[field].value_counts(normalize=True) * 
                          np.log2(result_df_s3[field].value_counts(normalize=True)))
        gen_entropy = -sum(result_df_s3[output].value_counts(normalize=True) * 
                         np.log2(result_df_s3[output].value_counts(normalize=True)))
        entropy_loss = (orig_entropy - gen_entropy) / orig_entropy * 100
        info_loss_data.append(('Strategy 3: Component', orig_entropy, gen_entropy, entropy_loss))

# Strategy 4: Relative
if 'result_df_s4' in globals():
    field = FIELD_CONFIG["strategy4_field"]
    output = f"{field}_relative"
    if output in result_df_s4.columns:
        orig_entropy = -sum(result_df_s4[field].value_counts(normalize=True) * 
                          np.log2(result_df_s4[field].value_counts(normalize=True)))
        gen_entropy = -sum(result_df_s4[output].value_counts(normalize=True) * 
                         np.log2(result_df_s4[output].value_counts(normalize=True)))
        entropy_loss = (orig_entropy - gen_entropy) / orig_entropy * 100
        info_loss_data.append(('Strategy 4: Relative', orig_entropy, gen_entropy, entropy_loss))

if info_loss_data:
    print(f"{'Strategy':<25} {'Original':<12} {'Generalized':<12} {'Loss %'}")
    print("-" * 70)
    for strategy, orig, gen, loss in info_loss_data:
        print(f"{strategy:<25} {orig:<12.4f} {gen:<12.4f} {loss:>6.1f}%")
    
    # Best balance
    best_idx = min(range(len(info_loss_data)), key=lambda i: info_loss_data[i][3])
    print(f"\n📊 Lowest info loss: {info_loss_data[best_idx][0]} ({info_loss_data[best_idx][3]:.1f}%)")
    
    highest_idx = max(range(len(info_loss_data)), key=lambda i: info_loss_data[i][3])
    print(f"📊 Highest info loss: {info_loss_data[highest_idx][0]} ({info_loss_data[highest_idx][3]:.1f}%)")

print("\n" + "=" * 80)

## Export Final Data

Export results from all 4 strategies

In [ ]:
print("=" * 80)
print("📦 EXPORTING FINAL DATA FROM ALL STRATEGIES")
print("=" * 80)

export_base_dir = task_dir
export_base_dir.mkdir(parents=True, exist_ok=True)

print(f"\n📂 Export directory: {export_base_dir}\n")

# Strategy 1: Rounding
if 'result_df_s1' in globals():
    print("=" * 80)
    print("📊 STRATEGY 1: ROUNDING")
    print("=" * 80)
    
    s1_dir = export_base_dir / 'strategy1_rounding'
    s1_dir.mkdir(parents=True, exist_ok=True)
    
    field = FIELD_CONFIG["strategy1_field"]
    output = f"{field}_rounded"
    
    export_cols = ['record_id', field, output, 'event_type', 'location', 'user_type', 'value']
    existing_cols = [c for c in export_cols if c in result_df_s1.columns]
    
    output_path = s1_dir / 'rounding_generalized_data.csv'
    try:
        result_df_s1[existing_cols].to_csv(output_path, index=False)
        print(f"\n✅ Saved: {output_path}")
        print(f"   Columns: {len(existing_cols)}, Rows: {len(result_df_s1):,}")
    except PermissionError:
        print(f"\n⚠️  Cannot save (file may be open): {output_path}")
    
    print(f"\n📊 Preview:")
    print(result_df_s1[existing_cols].head(3))

# Strategy 2: Binning
if 'result_df_s2' in globals():
    print("\n" + "=" * 80)
    print("📊 STRATEGY 2: BINNING")
    print("=" * 80)
    
    s2_dir = export_base_dir / 'strategy2_binning'
    s2_dir.mkdir(parents=True, exist_ok=True)
    
    field = FIELD_CONFIG["strategy2_field"]
    output = f"{field}_binned"
    
    export_cols = ['record_id', field, output, 'event_type', 'location', 'value']
    existing_cols = [c for c in export_cols if c in result_df_s2.columns]
    
    output_path = s2_dir / 'binning_generalized_data.csv'
    try:
        result_df_s2[existing_cols].to_csv(output_path, index=False)
        print(f"\n✅ Saved: {output_path}")
        print(f"   Columns: {len(existing_cols)}, Rows: {len(result_df_s2):,}")
    except PermissionError:
        print(f"\n⚠️  Cannot save (file may be open): {output_path}")
    
    print(f"\n📊 Preview:")
    print(result_df_s2[existing_cols].head(3))

# Strategy 3: Component
if 'result_df_s3' in globals():
    print("\n" + "=" * 80)
    print("📊 STRATEGY 3: COMPONENT")
    print("=" * 80)
    
    s3_dir = export_base_dir / 'strategy3_component'
    s3_dir.mkdir(parents=True, exist_ok=True)
    
    field = FIELD_CONFIG["strategy3_field"]
    output = f"{field}_component"
    
    export_cols = ['record_id', field, output, 'event_type', 'value']
    existing_cols = [c for c in export_cols if c in result_df_s3.columns]
    
    output_path = s3_dir / 'component_generalized_data.csv'
    try:
        result_df_s3[existing_cols].to_csv(output_path, index=False)
        print(f"\n✅ Saved: {output_path}")
        print(f"   Columns: {len(existing_cols)}, Rows: {len(result_df_s3):,}")
    except PermissionError:
        print(f"\n⚠️  Cannot save (file may be open): {output_path}")
    
    print(f"\n📊 Preview:")
    print(result_df_s3[existing_cols].head(3))

# Strategy 4: Relative
if 'result_df_s4' in globals():
    print("\n" + "=" * 80)
    print("📊 STRATEGY 4: RELATIVE")
    print("=" * 80)
    
    s4_dir = export_base_dir / 'strategy4_relative'
    s4_dir.mkdir(parents=True, exist_ok=True)
    
    field = FIELD_CONFIG["strategy4_field"]
    output = f"{field}_relative"
    
    export_cols = ['record_id', field, output, 'user_type', 'value']
    existing_cols = [c for c in export_cols if c in result_df_s4.columns]
    
    output_path = s4_dir / 'relative_generalized_data.csv'
    try:
        result_df_s4[existing_cols].to_csv(output_path, index=False)
        print(f"\n✅ Saved: {output_path}")
        print(f"   Columns: {len(existing_cols)}, Rows: {len(result_df_s4):,}")
    except PermissionError:
        print(f"\n⚠️  Cannot save (file may be open): {output_path}")
    
    print(f"\n📊 Preview:")
    print(result_df_s4[existing_cols].head(3))

# Summary
print("\n" + "=" * 80)
print("✅ EXPORT SUMMARY")
print("=" * 80)
print(f"\n📂 All files saved to: {export_base_dir}")
print(f"\n📁 Folder structure:")
print(f"   ├── strategy1_rounding/")
print(f"   │   └── rounding_generalized_data.csv")
print(f"   ├── strategy2_binning/")
print(f"   │   └── binning_generalized_data.csv")
print(f"   ├── strategy3_component/")
print(f"   │   └── component_generalized_data.csv")
print(f"   └── strategy4_relative/")
print(f"       └── relative_generalized_data.csv")

if all(v in globals() for v in ['elapsed_s1', 'elapsed_s2', 'elapsed_s3', 'elapsed_s4']):
    total_time = elapsed_s1 + elapsed_s2 + elapsed_s3 + elapsed_s4
    print(f"\n⏱️  Total processing time: {total_time:.2f}s")

print("\n" + "=" * 80)
print("🎉 EXPORT COMPLETE!")
print("=" * 80)

## 🎯 Final Summary

**Accomplished:**
- ✅ Strategy 1: Rounding (day-level precision)
- ✅ Strategy 2: Binning (business hours categorization)
- ✅ Strategy 3: Component (year-month extraction)
- ✅ Strategy 4: Relative (temporal distance expressions)
- ✅ Privacy metrics calculation (k-anonymity)
- ✅ Information loss analysis (entropy)
- ✅ Performance comparison
- ✅ Multi-stage pipeline execution

**Key takeaways:**
- **Rounding**: Best for preserving temporal ordering while reducing precision
- **Binning**: Converts continuous time to discrete categories, highest privacy
- **Component**: Useful for keeping specific date parts (year-month)
- **Relative**: Human-readable temporal expressions, very high privacy
- All strategies provide different privacy/utility trade-offs
- Choose based on your specific use case and privacy requirements

**Strategy Selection Guide:**
- Need exact ordering? → **Rounding**
- Need categorical time periods? → **Binning**
- Need monthly/yearly aggregation? → **Component**
- Need maximum privacy? → **Relative**

**Next steps:**
- Experiment with different parameters (rounding_unit, bin_type, components)
- Try your own data
- Combine with other anonymization operations
- Deploy to production pipelines
- Test different privacy thresholds

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)
- 🎓 [Advanced DateTime Guide](../../docs/advanced_datetime.md)
- 🕐 [Temporal Privacy Metrics](../../docs/temporal_privacy.md)